This notebook was created by Donna Faith Go.

In [1]:
# import sys
# !{sys.executable} -m pip install -qq -r requirements.txt

In [2]:
# standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import re

# knn imputer
from sklearn.impute import KNNImputer

# granger causality
from statsmodels.tsa.stattools import grangercausalitytests

# ignore warnings
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

## Applying Granger Causality

## Access Data

In [3]:
filepath = r'SP500 comapnies list.pkl'
with open(filepath, 'rb') as f:
    stocks_info = pickle.load(f)
stocks_info.set_index('Security', inplace=True)
stocks_info

,Symbol,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
Security,,,,,,,
3M,MMM,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
A. O. Smith,AOS,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
Abbott Laboratories,ABT,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
AbbVie,ABBV,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
Accenture,ACN,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...
Xylem Inc.,XYL,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
Yum! Brands,YUM,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
Zebra Technologies,ZBRA,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969


In [4]:
directory = r'data'
closing_dict = {}

for name in sorted(os.listdir(directory)):
    if name == '.ipynb_checkpoints':
        continue

    # getting the ticker 
    firm = name.split('.pkl')[0]
    if firm in stocks_info.index:
        symbol = stocks_info.loc[firm, 'Symbol']

    # getting the file
    filepath = os.path.join(directory, name)
    data = pd.read_pickle(filepath)
    data.columns = data.columns.get_level_values(0)
    data = data[['Close', 'High', 'Low', 'Open', 'Volume']]
    data.columns.name = None

    # getting the closing prices
    closing = data.loc[:, 'Close']
    closing = closing[~closing.index.duplicated(keep='first')]
    closing_dict[symbol] = closing

# creating a dataframe
closing_df = pd.concat(closing_dict, axis=1)
closing_df.sort_index(inplace=True)

# dropping columns that have all nulls
closing_df.dropna(how='all', axis=1, inplace=True)
original_closing = closing_df.copy()

In [5]:
closing_df.head(5)

,MMM,AOS,AES,APA,T,ABBV,ABT,ACN,ADBE,AMD,...,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS,EBAY
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,19.257389,2.254067,23.135981,10.113912,6.250895,NaN,8.093055,NaN,16.274677,15.500,...,NaN,NaN,NaN,6.577461,NaN,4.503591,25.027779,NaN,NaN,6.613386
2000-01-04,18.492201,2.221588,22.218521,9.669347,5.885149,NaN,7.861829,NaN,14.909399,14.625,...,NaN,NaN,NaN,6.728915,NaN,4.413068,24.666668,NaN,NaN,5.993016
2000-01-05,19.027836,2.215093,22.457857,9.947192,5.976584,NaN,7.847377,NaN,15.204176,15.000,...,NaN,NaN,NaN,6.988552,NaN,4.435700,25.138889,NaN,NaN,6.393916
2000-01-06,20.558224,2.182613,22.637363,10.891903,5.860783,NaN,8.121960,NaN,15.328288,16.000,...,NaN,NaN,NaN,6.923641,NaN,4.397984,23.777779,NaN,NaN,6.314907
2000-01-07,20.966341,2.273555,23.076149,10.854856,5.911020,NaN,8.208676,NaN,16.072985,16.250,...,NaN,NaN,NaN,6.923641,NaN,4.299915,23.513889,NaN,NaN,6.309053


## Data Preprocessing

### Log Returns

In [6]:
# get log returns
log_returns = np.log(closing_df / closing_df.shift(1))
log_returns.dropna(how='all', axis=0, inplace=True)
log_returns = log_returns.loc['2001':]

### K-NN Imputer

In [7]:
# create model
imputer = KNNImputer(n_neighbors=5)
transformed_array = imputer.fit_transform(log_returns)

# create dataframe
log_returns_transformed = pd.DataFrame(
    transformed_array, 
    columns=log_returns.columns,
    index=log_returns.index
)

## Shock Periods
We defined a shock period based on deviations from the long-term mean of daily logarithmic returns for the S&P 500 Index. 
Specifically, a shock period was identified if the absolute value of the average daily logarithmic return during a potential shock window (e.g., 5 consecutive trading days) exceeded 2 standard deviations from the long-term mean.

In [8]:
filepath = r'sp 500 data.pkl'
with open(filepath, 'rb') as f:
    gspc_data = pickle.load(f)

# get log returns 
gspc_returns = np.log(gspc_data / gspc_data.shift(1))
gspc_returns.dropna(how='all', axis=0, inplace=True)

# get long term mean and std
gspc_long_mean = gspc_returns['^GSPC'].mean()
gspc_long_std = gspc_returns['^GSPC'].std()
print(f"Long-term mean: {gspc_long_mean}")
print(f"Long-term std: {gspc_long_std}")

Long-term mean: 0.00017664233488167506
Long-term std: 0.01248281996701577


In [ ]:
len(gspc_data) * 0.1 * 0.1

In [9]:
# window size
window_size = 5

# average returns per day
rolling_avg_returns = gspc_returns['^GSPC'].rolling(window=window_size).mean()

# shock periods
shock_windows = abs(rolling_avg_returns - gspc_long_mean) > (2 * gspc_long_std)
shock_dates = shock_windows[shock_windows].index
print(f"Total shock windows found: {len(shock_dates)}")

Total shock windows found: 17


In [ ]:
# get boundaries
gspc_returns['2 sigma std'] = 2 * gspc_long_std
gspc_returns['neg 2 sigma std'] = -2 * gspc_long_std
gspc_returns

# plotting
plt.figure(figsize=(15, 6))
plt.plot(gspc_returns['^GSPC'], label='returns')
plt.plot(gspc_returns['2 sigma std'], color='red', label='sigma')
plt.plot(gspc_returns['neg 2 sigma std'], color='red', label='sigma')
plt.plot(rolling_avg_returns, label='rolling avg')
plt.legend()
plt.ylabel("Log Returns")
plt.xlabel("Dates")
plt.title("Shock Periods")
plt.show()

## Granger Causality
Note: 
1. Using `window=5` LEADS TO InfeasibleTestError: The Granger causality test statistic cannot be computed because the VAR has a perfect fit of the data.

In [10]:
def get_adjacency_matrices(
    log_returns_transformed, window=5, 
    overwrite=True, output_dir=r'adjacency matrices'
):
    os.makedirs(output_dir, exist_ok=True)
    
    # look at each window
    for end_window in range(window, len(log_returns_transformed.index)):
        # define start and end windows
        start_window = window - window
        
        # for creating the file
        date_str = str(log_returns_transformed.index[end_window]).split(" ")[0]
        filepath = f"{output_dir}/{date_str}.pkl"

        # check if file already exists
        if not overwrite and os.path.exists(filepath):
            print(f"Skipped {date_str}")
            continue
    
        # get from dataframe
        temp_df = log_returns_transformed[start_window:end_window]

        # granger causality matrix
        adj_matrix = np.zeros((len(temp_df.columns), len(temp_df.columns)))
        for i, col1 in enumerate(temp_df.columns):
            for j, col2 in enumerate(temp_df.columns):
                if i != j:
                    try:
                        test_result = grangercausalitytests(
                            temp_df[[col1, col2]], 
                            maxlag=[1], 
                            verbose=False
                        )[1][0]['ssr_chi2test'][1]
                    
                        adj_matrix[i, j] = 1 if test_result < 0.05 else 0
                    
                    except Exception:
                        adj_matrix[i, j] = np.nan
    
        # save adjacecny matrix
        new_data = {
            'date': date_str,
            'adjacency matrix': adj_matrix
        }
        with open(f"{output_dir}/{date_str}.pkl", 'wb') as f:
            pickle.dump(new_data, f)
        print(f"Saved: {date_str}!")

In [ ]:
get_adjacency_matrices(log_returns_transformed, overwrite=False, window=5)

KeyboardInterrupt: 

## Parallel processing

## Supplementary

### Unused Code

In [ ]:
# # set window
# window = 10

# # get the windows
# for window in range(window, len(log_returns_transformed.index)):
#     # define start and end windows
#     start_window = window - 5
#     end_window = window

#     # get from dataframe
#     temp_df = log_returns_transformed[start_window:end_window]

#     # granger causality matrix
#     adj_matrix = pd.DataFrame(index=temp_df.columns, columns=temp_df.columns, dtype=float)
#     for i, col1 in enumerate(temp_df.columns):
#         for col2 in temp_df.columns[i+1:]:
#             test_result = grangercausalitytests(
#                 temp_df[[col1, col2]], 
#                 maxlag=[1], 
#                 verbose=False
#             )[1][0]['ssr_chi2test'][1]
#             adj_matrix.loc[col1, col2] = adj_matrix.loc[col2, col1] = 1 if test_result < 0.05 else 0

#     # get date
#     date_str = str(log_returns_transformed.index[end_window]).split(" ")[0]

#     # save adjacecny matrix
#     output_dir = r'adjacency matrices'
#     adj_matrix.to_pickle(f"{output_dir}/{date_str}.pkl")
#     print(f"Saved: {date_str}!")